# Activity Classification: Walking vs Running

---

## Table of Contents
1. Problem Statement
2. Objective
3. Key Outcomes
4. Data Loading & Cleaning
5. Exploratory Data Analysis
6. Model Training
7. Model Comparison
8. Download Model Performance Report
9. Hyperparameter Tuning
10. Conclusions

---

## Problem Statement
Develop a predictive model to classify whether a person is **walking** or **running** using sensor-based features such as acceleration and gyroscope readings.

---

## Objective
- Build a classification model using sensor data.
- Evaluate multiple algorithms for accuracy and reliability.

---

## Key Outcomes
- **MLP Neural Network achieved the highest accuracy (99.08%)**.
- Random Forest and Decision Tree also performed well (>98%).
- Logistic Regression and SVM were less accurate (~86%).

---

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [15]:
class ActivityClassifierPipeline:
    def __init__(self, filepath):
        self.filepath = filepath
        self.df = None
        self.X_train, self.X_test, self.y_train, self.y_test = None, None, None, None
        self.results = {}

    def load_data(self):
        self.df = pd.read_csv(self.filepath)
        print("✅ Data Loaded Successfully")

    def clean_data(self):
        self.df.drop(columns=['username'], inplace=True)
        numeric_cols = ['wrist','acceleration_x','acceleration_y','acceleration_z','gyro_x','gyro_y','gyro_z']
        for col in numeric_cols:
            self.df[col] = np.cbrt(self.df[col])
        self.df['activity'] = self.df['activity'].replace({0: 'walk', 1: 'run'})
        print("✅ Data Cleaning Completed")

    def exploratory_analysis(self):
        selected_features = ['acceleration_x','acceleration_y','acceleration_z','gyro_x','gyro_y','gyro_z','activity']
        sns.pairplot(self.df[selected_features], hue='activity', diag_kind='kde')
        plt.savefig("eda_plots_pairplot.png")
        plt.close()
        corr = self.df[selected_features[:-1]].corr()
        sns.heatmap(corr, annot=True, cmap='coolwarm')
        plt.savefig("eda_plots_corr.png")
        plt.close()
        print("✅ EDA plots exported")

    def prepare_data(self):
        X = self.df[['acceleration_x','acceleration_y','acceleration_z','gyro_x','gyro_y','gyro_z']]
        y = self.df['activity']
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=10)
        print("✅ Data Prepared for Modeling")

    def train_models(self):
        models = {
            "Logistic Regression": LogisticRegression(),
            "Decision Tree": DecisionTreeClassifier(random_state=42),
            "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
            "SVM": SVC(kernel='linear', C=1.0, random_state=42),
            "MLP": MLPClassifier(max_iter=500)
        }
        for name, model in models.items():
            model.fit(self.X_train, self.y_train)
            y_pred = model.predict(self.X_test)
            acc = accuracy_score(self.y_test, y_pred)
            self.results[name] = acc
        print("✅ Models Trained Successfully")

    def compare_results(self):
        best_model = max(self.results, key=self.results.get)
        print("\nModel Performance:")
        for model, acc in self.results.items():
            print(f"{model}: {acc:.4f}")
        print(f"\n✅ Best Model: {best_model} with accuracy {self.results[best_model]:.4f}")
        plt.figure(figsize=(8, 5))
        sns.barplot(x=list(self.results.keys()), y=list(self.results.values()), hue=list(self.results.keys()), palette="viridis", legend=False)
        plt.title("Model Accuracy Comparison")
        plt.ylabel("Accuracy")
        plt.xticks(rotation=15)
        for i, acc in enumerate(self.results.values()):
            plt.text(i, acc + 0.005, f"{acc:.2f}", ha='center', fontsize=10)
        plt.tight_layout()
        plt.savefig("model_comparison.png")
        plt.close()
        print("✅ Model comparison chart exported")

    def export_results_csv(self):
        df_results = pd.DataFrame(list(self.results.items()), columns=['Model', 'Accuracy'])
        df_results.to_csv('model_performance.csv', index=False)
        print("✅ Model performance exported to model_performance.csv")

In [17]:
pipeline = ActivityClassifierPipeline(filepath='walkrun.csv')
pipeline.load_data()
pipeline.clean_data()
pipeline.exploratory_analysis()
pipeline.prepare_data()
pipeline.train_models()
pipeline.compare_results()
pipeline.export_results_csv()

✅ Data Loaded Successfully
✅ Data Cleaning Completed
✅ EDA plots exported
✅ Data Prepared for Modeling
✅ Models Trained Successfully

Model Performance:
Logistic Regression: 0.8563
Decision Tree: 0.9827
Random Forest: 0.9897
SVM: 0.8680
MLP: 0.9901

✅ Best Model: MLP with accuracy 0.9901
✅ Model comparison chart exported
✅ Model performance exported to model_performance.csv


In [18]:
from IPython.display import FileLink
FileLink('model_performance.csv')

/Users/lz935c/Library/CloudStorage/OneDrive-GeneralMotors/GM/Adhocs/AIE Classwork/internshipProject/model_performance.csv

In [23]:
param_dist = {
    'n_estimators': np.arange(100, 500, 50),
    'max_depth': [None] + list(np.arange(10, 50, 10)),
    'min_samples_split': np.arange(2, 11),
    'min_samples_leaf': np.arange(1, 11)
}
rf = RandomForestClassifier(random_state=42)
random_search = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=10, cv=3, scoring='accuracy', random_state=42, n_jobs=-1)
random_search.fit(pipeline.X_train, pipeline.y_train)
print("Best Parameters:", random_search.best_params_)
print("Best Accuracy:", random_search.best_score_)

Best Parameters: {'n_estimators': 150, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_depth': 30}
Best Accuracy: 0.9886214837224004


In [25]:
param_dist_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50,50), (100,50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
    'alpha': [0.0001, 0.001, 0.01]
}
mlp = MLPClassifier(max_iter=500)
random_search_mlp = RandomizedSearchCV(mlp, param_distributions=param_dist_mlp, n_iter=10, cv=3, scoring='accuracy', random_state=42, n_jobs=-1)
random_search_mlp.fit(pipeline.X_train, pipeline.y_train)
print("Best Parameters:", random_search_mlp.best_params_)
print("Best Accuracy:", random_search_mlp.best_score_)

Best Parameters: {'solver': 'adam', 'hidden_layer_sizes': (50, 50), 'alpha': 0.001, 'activation': 'relu'}
Best Accuracy: 0.9898105085715145


# Result Summary for Predictive Model

## ✅ Data Processing & Modeling
- Data was successfully loaded and cleaned.
- Exploratory Data Analysis (EDA) was performed, and visual insights were exported.
- Predictor variables were standardized, and multiple classification models were trained.

---

## Model Performance Overview

| Model                | Accuracy   |
|----------------------|-----------|
| Logistic Regression  | **85.63%** |
| Decision Tree        | **98.27%** |
| Random Forest        | **98.97%** |
| SVM                  | **86.80%** |
| MLP (Neural Network) | **99.01%** |

*(See the charts in the graph folder with name `model_comparison.png` for visual comparison.)*

---

## Key Insights
- **MLP (Multi-Layer Perceptron)** achieved the highest accuracy at **99.08%**, making it the best-performing model for this classification task.
- **Random Forest** and **Decision Tree** also performed exceptionally well, both above **98% accuracy**, indicating strong predictive capability.
- **Logistic Regression** and **SVM** provided reasonable performance but were less effective compared to tree-based and neural network models.

---

## Conclusion
The **MLP Neural Network** is the most suitable model for classifying walking vs. running activities based on the given sensor data.  
Its superior accuracy suggests that **deep learning approaches can effectively capture complex patterns in motion data**.